In [0]:
from pyspark.sql.functions import current_date, current_timestamp

In [0]:
df = spark.range(2).alias("id")

In [0]:
df = df.withColumns(
    {
        "current_date" :current_date(),
        "current_timestamp":current_timestamp()

    }
)

In [0]:
df.display()


####2. Requirement
- You are given the below dataframes.
- Convert the df_1 to timestamp

In [0]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")]
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))

In [0]:
from pyspark.sql.functions import try_to_timestamp,col,lit
df_1 = df_1.withColumn(
    "timestamp_format",
     try_to_timestamp(
         col("string_time"), 
         lit("yyyy-MM-dd'T'HH:mm:ss.SSSS")))

In [0]:
df_1.display()

2.2 Convert the df_2 to time

In [0]:
df_2.display()

In [0]:
# to_timestamp, #extract hr,min,ss and then individual components and then date_format


from pyspark.sql.functions import try_to_timestamp,to_timestamp

df2 = df_2.withColumn('to_timestamp_format',
     to_timestamp(col('string_time'),"dd-MM-yyyy HH:mm:ss.SSSS"))

In [0]:
from pyspark.sql.functions import split, hour,min, second, date_format,lit,col,minute,second,day

df2 = df2.withColumn("date_format_time",
     date_format(
         col("to_timestamp_format"), "HH:mm:ss.SSSS"))


In [0]:
df2.display()

In [0]:
#extract parts of data

df2 = df2.withColumns(

    {
  "hour_part":hour(col("to_timestamp_format")),

  "minute_part": minute(col("to_timestamp_format")),

  "second_part" :second(col("to_timestamp_format")),

  "day_part" : day(col("to_timestamp_format"))

    }
)

In [0]:
df2.display()

####3. Timezone information

1. A timestamp without timezone information is incomplete.
2. Spark offers two data types for timestamp
    1. TIMESTAMP
    2. TIMESTAMP_NTZ
3. For TIMESTAMP, Spark assumes session timezone as the default when timezone is not specified
4. Session timezone is specified as spark.sql.session.timeZone

In [0]:
#default session timezone
spark.conf.get("spark.sql.session.timeZone")

In [0]:
# set the timezone
spark.conf.set("spark.sql.session.timeZone", "Etc/UTC")

####4. Working with NTZ data

In [0]:
events_no_tz_schema = "component STRING,event_time STRING, reading STRING"

data_events =  spark.read.format("csv").schema(events_no_tz_schema).option("header", "true").load("/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-no-tz.csv")

In [0]:
data_events.display()

In [0]:
# from pyspark.sql.functions import to_timestamp,col,lit
# data_events = data_events.withColumn("event_time", to_timestamp(col("event_time"),lit("dd-MM-yyyy HH:mm:ss.SSSS")))

In [0]:
data_events.display()

In [0]:
from pyspark.sql.functions import to_timestamp,col,lit,to_timestamp_ntz
data_events = data_events.withColumns(
    {

    "event_time_tz": to_timestamp(
                        col("event_time"),lit("dd-MM-yyyy HH:mm:ss.SSSS")),
    
    "event_timestamp_ntz" :to_timestamp_ntz (
                            col("event_time"),lit("dd-MM-yyyy HH:mm:ss.SSSS"))
           


    }
)


In [0]:
# set the timezone
spark.conf.set("spark.sql.session.timeZone", "IST")

In [0]:
data_events.display()

In [0]:
from pyspark.sql.functions import to_timestamp,col,lit,to_timestamp_ntz,convert_timezone,lit
data_events = data_events.withColumn(
    'timezone_convert_Aus_Indian',
    convert_timezone(
            lit('Australia/Sydney'),lit('Asia/Kolkata'),col('event_timestamp_ntz')))

In [0]:
%sql
 SELECT convert_timezone('Australia/Sydney', 'IST', current_timestamp());
 

-- Current session local timezone: UTC
-- SELECT convert_timezone('America/Los_Angeles', timestamp_ntz'2021-12-06 08:00:00');
 --2021-12-06 00:00:00

In [0]:
%sql
 SELECT convert_timezone('Australia/Sydney',
 'Asia/Kolkata',timestamp_ntz'2026-06-05 20:17:00'

 )

In [0]:
#convert to 